# Kapitel 5 - Dimensionsreducering

## Faktafrågor

### 1. Vad menas med curse of dimensionality.

Curse of dimensionality är problem som uppstår när man har för många variabler i datan. Datan blir mer gles, punkterna hamnar långt från varandra och det blir svårare för modellen att hitta mönster. Man behöver också mycket mer data för att det ska funka bra, annars är risken större att modellen överanpassar.

### 2. Vad är dimensionsreducering och varför görs det?

Dimensionsreducering är kopplat till förra frågan, det är ett sätt att minska antalet variabler i datan men ändå behålla det mesta av informationen. Görs bland annat för att slippa curse of dimensionality och för att modellträningen ska gå snabbare.

### 3. Förklara översiktligt hur PCA fungerar. Använd figur 5.4 på sidan 224 i din förklaring.

PCA väljer bland olika möjliga linjer den som sprider ut prickarna mest (c1 i figuren), det bevarar mest information. Linjer som c2 klumpar ihop prickarna mycket tätare, mindre spridning kvar, och det blir sämre resultat eftersom mer information går förlorad.

## Resonemangfrågor

### 5. Stina påstår att man alltid vill ha modeller som genomför så bra prediktioner som möjligt. Kalle påstår att tid också är en viktig aspekt. Vad säger du?

Jag tror Kalle har rätt, man måste balansera mellan tillräckligt bra precision och tid, och kanske även computer power som räknas in i kalkylen. Dock vill man ibland ha precision som viktigaste momentet oavsett tid eller pris, så det beror helt enkelt på use case.

### 6. Efter att vi genomfört en PCA, vad händer med tolkningen av variablerna?

Modellen blir snabbare, men tolkningen av variablerna blir svårare. De nya variablerna (principalkomponenterna) är blandningar av dom gamla variablerna, så dom har ingen tydlig verklig betydelse längre, typ "ålder" eller "inkomst" blir bara abstrakta tal istället.

## Koduppgifter

### 8. Förklara vad PCA-koden gör (skapar data, reducerar till 2D, återskapar till 3D med inverse_transform).

In [1]:
import numpy as np
from sklearn.decomposition import PCA

X = np.random.rand(1000, 3)
print(X[0:5])

pca = PCA(n_components=2)
X2D = pca.fit_transform(X)
print(X2D[0:5])

X3D_inv = pca.inverse_transform(X2D)
print(np.allclose(X3D_inv, X))

[[0.88598516 0.26040711 0.63653682]
 [0.38879505 0.44941289 0.85909537]
 [0.92100737 0.69357762 0.95438446]
 [0.12303105 0.89267626 0.40271991]
 [0.58213961 0.19614462 0.84131826]]
[[-0.12699889  0.33053038]
 [-0.3116556   0.14936628]
 [-0.01366188  0.65268906]
 [ 0.18797328 -0.24737372]
 [-0.40765826  0.22647679]]
False


Koden skapar slumpmässig data med 3 kolumner, reducerar den till 2 kolumner med PCA, och försöker sen återskapa 3 kolumner igen med inverse_transform. Resultatet blir False eftersom information gick förlorad när vi gick från 3D till 2D, så vi kan inte få tillbaka exakt samma data, bara en approximation.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.decomposition import PCA

df = pd.read_csv('data/car_price_dataset.csv', sep=';')
df = df[df['Brand'].isin(['Toyota', 'BMW'])]

df_model = df.drop(columns=['Model'])
df_model = pd.get_dummies(df_model, columns=['Brand', 'Fuel_Type', 'Transmission'], drop_first=True)

X = df_model.drop(columns=['Price'])
y = df_model['Price']

X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=40)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.3, random_state=36)

print(X_train.shape)

(1102, 11)


#### Utan PCA (som i kapitel 3)

In [3]:
lr = LinearRegression()
lr.fit(X_train, y_train)

rmse_no_pca = root_mean_squared_error(y_val, lr.predict(X_val))
print("RMSE utan PCA:", rmse_no_pca)

RMSE utan PCA: 105.55978171532172


#### Med PCA

Vi kör PCA på träningsdatan (fit bara på train), och transformerar sen val/test med samma PCA.

In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

pca = PCA(n_components=0.95)  # behåll 95% av variansen
X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca = pca.transform(X_val_scaled)

print("Antal kolumner innan:", X_train.shape[1])
print("Antal kolumner efter PCA:", X_train_pca.shape[1])

lr_pca = LinearRegression()
lr_pca.fit(X_train_pca, y_train)

rmse_pca = root_mean_squared_error(y_val, lr_pca.predict(X_val_pca))
print("RMSE med PCA:", rmse_pca)

Antal kolumner innan: 11
Antal kolumner efter PCA: 10
RMSE med PCA: 389.08450576762493


RMSE blev mycket sämre med PCA (389 mot 105). PCA vet inte vilken variabel som faktiskt är viktig för priset, den bryr sig bara om spridning, så viktig info kan gå förlorad.